# Modelo de Traslado de Valores - FINABIEN

## 0.2 Análisis Estadístico

### El propósito de este 'notebook' es analizar las características estadísticas que presentan cada una de las sucursales para observar situaciones particulares que se pudieran mejorar. 

In [1]:
import pandas as pd
import numpy as np
import pickle
import Funciones as f
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

In [ ]:
# Ruta donde se guardó el archivo de los días festivos
ruta_archivo_flujo_caja = "C:/Users/jose.luis.flores/Documents/Repositorios/MTDV/2. Datos/df_mod.pkl"

with open(ruta_archivo_flujo_caja, "rb") as archivo:
    df_mod = pickle.load(archivo)

print("Base de datos cargada correctamente.")

In [ ]:
# 1. Filtrar sucursales que NO terminan en '000'
df_filtrado = df_mod[~df_mod['id_sucursal'].astype(str).str.endswith('000')].copy()

# 2. Extraer el año de la fecha
df_filtrado['año'] = df_filtrado['fecha'].dt.year

# 3. Filtrar solo 2023, 2024, 2025
df_filtrado = df_filtrado[df_filtrado['año'].isin([2023, 2024, 2025])]

# 4. Calcular estadísticas por sucursal y año
estadisticas_por_sucursal = (
    df_filtrado
    .groupby(['año', 'id_sucursal'])['saldo_final']
    .describe(percentiles=[0.25, 0.5, 0.75])
    .unstack(level=0)  # Pivotear años como columnas
    .swaplevel(axis=1)  # Ordenar jerarquía de columnas
    .sort_index(axis=1, level=0)
)

# 5. Exportar a Excel
estadisticas_por_sucursal.to_excel('estadisticas_saldo_final_por_sucursal.xlsx')

print("✅ Reporte generado en 'estadisticas_saldo_final_por_sucursal.xlsx'")
print(estadisticas_por_sucursal.head())  # Mostrar un preview

In [ ]:
# Filtrar sucursales que NO terminan en '000'
df_filtrado = df_mod[~df_mod['id_sucursal'].astype(str).str.endswith('000')].copy()

# Extraer año y mes de la fecha
df_filtrado['año'] = df_filtrado['fecha'].dt.year
df_filtrado['mes'] = df_filtrado['fecha'].dt.month

# Calcular promedios mensuales por sucursal
df_promedios = df_filtrado.groupby(['año', 'id_sucursal', 'mes']).agg({
    'flujo_efectivo': 'mean',
    'entradas': 'mean',
    'salidas': 'mean'
}).reset_index()

# Luego calcular el promedio anual (promedio de los promedios mensuales)
df_anual = df_promedios.groupby(['año', 'id_sucursal']).agg({
    'flujo_efectivo': 'mean',
    'entradas': 'mean',
    'salidas': 'mean'
}).reset_index()

# Definir los rangos
rangos = [
    ('Más de 200,000', lambda x: x > 200000),
    ('Más de 50,000 pero menos que 200,000', lambda x: (x > 50000) & (x <= 200000)),
    ('Menos de 50,000 pero más que 0', lambda x: (x > 0) & (x <= 50000)),
    ('Menos de 0 pero más que -50,000', lambda x: (x < 0) & (x >= -50000)),
    ('Menos que -50,000 pero más que -200,000', lambda x: (x < -50000) & (x >= -200000)),
    ('Menos que -200,000', lambda x: x < -200000)
]

# Función para generar el reporte por año
def generar_reporte_por_año(df, año):
    df_año = df[df['año'] == año]
    reporte = {
        'Flujo de Efectivo': {desc: df_año['flujo_efectivo'][cond(df_año['flujo_efectivo'])].count() 
                             for desc, cond in rangos},
        'Entradas': {desc: df_año['entradas'][cond(df_año['entradas'])].count() 
                     for desc, cond in rangos},
        'Salidas': {desc: df_año['salidas'][cond(df_año['salidas'])].count() 
                    for desc, cond in rangos}
    }
    return pd.DataFrame(reporte)

# Obtener años únicos
años = df_anual['año'].unique()

# Crear un ExcelWriter para exportar múltiples hojas
with pd.ExcelWriter('reporte_sucursales_anual.xlsx') as writer:
    for año in sorted(años):
        df_reporte = generar_reporte_por_año(df_anual, año)
        df_reporte.to_excel(writer, sheet_name=f'Año {año}', index_label='Rango')
        
    # Agregar una hoja con los promedios por sucursal
    df_anual.to_excel(writer, sheet_name='Datos Completos', index=False)

print("Reporte generado exitosamente en 'reporte_sucursales_anual.xlsx'")

In [ ]:
df_mod.head()

In [ ]:
# Muestra el número de sucursales en la base de datos y el tipo de datos
print("Número de sucursales FINABIEN:", df_mod['id_sucursal'].nunique())

print("La base de datos contiene filas y columnas:",df_mod.shape)

print(df_mod.dtypes)

In [ ]:
# Muestra la lista de todos los estados cargados en la base de datos

print("Número de Gerencias FINABIEN:", df_mod['nombre_estado'].nunique())

print(list(df_mod['nombre_estado'].unique()))

In [ ]:
# Ejemplo para el estado seleccionado:
estado_seleccionado = "guerrero"  # Cambia esto por el estado que desees filtrar
flujo_estado = f.filtrar_estado(df_mod, estado_seleccionado)

# Mostrar las primeras filas del resultado
flujo_estado.head(10)

In [ ]:
conteo_mayores_cero = flujo_estado.groupby('id_sucursal')[['remesas_entrada', 'remesas_salida']].apply(
    lambda x: (x > 0).sum()
)

# Ordenar por remesas_entrada de mayor a menor
conteo_mayores_cero = conteo_mayores_cero.sort_values(by='remesas_entrada', ascending=False)

# Mostrar las primeras y últimas filas
print(conteo_mayores_cero.head())
print(conteo_mayores_cero.tail())

In [ ]:
# Ejemplo de uso
ids_disponibles = f.obtener_ids_sucursales(flujo_estado)

print("Número de sucursales FINABIEN en el estado:", flujo_estado['id_sucursal'].nunique())

print("Sucursales disponibles:", ids_disponibles)

In [ ]:
# Ejemplo de uso:
id_sucursal_deseada = "12003"  # Reemplaza con el ID de la sucursal que quieres filtrar
flujo_sucursal = f.obtener_datos_sucursal(flujo_estado, id_sucursal_deseada)

# Mostrar los primeros registros para verificar
flujo_sucursal.head(15)

In [ ]:
# Crear una nueva columna 'mes' con el primer día del mes (por simplicidad)
flujo_sucursal['mes'] = flujo_sucursal['fecha'].dt.to_period('M').dt.to_timestamp()

# Agrupar por mes y contar cuántos valores > 0 hay en cada columna
remesas_por_mes = flujo_sucursal.groupby('mes')[['remesas_entrada', 'remesas_salida']].apply(
    lambda x: (x > 0).sum()
)

print(remesas_por_mes)

In [ ]:
conteo_mayores_cero = flujo_sucursal.groupby('id_sucursal')[['remesas_entrada', 'remesas_salida']].apply(
    lambda x: (x > 0).sum()
)

conteo_mayores_cero.head(15)

In [ ]:
# Asegúrate de que la columna 'fecha' sea de tipo datetime
flujo_sucursal['fecha'] = pd.to_datetime(flujo_sucursal['fecha'])

# Ordenar por fecha (opcional pero recomendable para series temporales)
flujo_sucursal = flujo_sucursal.sort_values(by='fecha')

# Crear la gráfica
plt.figure(figsize=(12, 6))
plt.plot(flujo_sucursal['fecha'], flujo_sucursal['remesas_salida'], marker='o', linestyle='-')
plt.title('Remesas de Salida por Fecha')
plt.xlabel('Fecha')
plt.ylabel('Remesas de Salida')
plt.grid(True)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
flujo_sucursal.describe()

In [ ]:
# Ejemplo de uso:
id_sucursal_seleccionada = "15012"  # Asegúrate de pasar el ID como string si es un object
f.graficar_sucursal(flujo_estado, id_sucursal_seleccionada)

In [ ]:
# 🔍 Descomposición de la serie temporal
decomposition = seasonal_decompose(flujo_sucursal['flujo_efectivo'], model='additive', period=30)
decomposition.plot()
plt.show()

In [ ]:
# Graficar ACF y PACF
fig, axes = plt.subplots(1, 2, figsize=(16,5))
plot_acf(flujo_sucursal['flujo_efectivo'].dropna(), lags=40, ax=axes[0])
plot_pacf(flujo_sucursal['flujo_efectivo'].dropna(), lags=40, ax=axes[1])
plt.show()

In [ ]:
def adf_test(series):
    result = adfuller(series.dropna())
    print(f"Estadístico ADF: {result[0]}")
    print(f"Valor p: {result[1]}")
    print("Conclusión:", "La serie es estacionaria" if result[1] < 0.05 else "La serie NO es estacionaria")

# Aplicamos la prueba
adf_test(flujo_sucursal['flujo_efectivo'])

In [ ]:
# Ruta donde se guardará el archivo
ruta_archivo_flujo_efectivo_sucursal = "C:/Users/jose.luis.flores/Documents/Personal/Certificados/Machine Learning/3. MTDV - FINABIEN/2. Datos/flujo_sucursal.pkl"

# Guardar el DataFrame en un archivo pickle
with open(ruta_archivo_flujo_efectivo_sucursal, "wb") as archivo:
    pickle.dump(flujo_sucursal, archivo)

print(f"Base de datos guardada en {ruta_archivo_flujo_efectivo_sucursal}")

In [ ]:
# Ruta donde se guardará el archivo
ruta_archivo_flujo_efectivo_estado = "C:/Users/jose.luis.flores/Documents/Repositorios/MTDV/2. Datos/flujo_estado.pkl"

# Guardar el DataFrame en un archivo pickle
with open(ruta_archivo_flujo_efectivo_estado, "wb") as archivo:
    pickle.dump(flujo_estado, archivo)

print(f"Base de datos guardada en {ruta_archivo_flujo_efectivo_estado}")